# Analyze Track Library to Produce Chromagram Vectors

For each track in the given track library path list, this notebook produces a chromagram vector for each track, per time step within each track.

Each chromagram vector covers the entire piano keyboard, per note. Stated more technically, each chromagram vector covers each note within the scientific pitch notation range C0 through B8. 

## Load useful libraries

In [1]:
import librosa
import multiprocessing
import pandas as pd

In [2]:
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
import pyspark.sql.functions as F

In [3]:
from chromagram_functions import process_octave

## User settings

In [4]:
output_directory = 'output'
sampling_rate = 22050
hop_length = 512 * 100
spark_memory = '70G'

# location of tracks to analyze for building the track feature database
path_library_parquet = '../database/music/output/playlist.parquet'

## Read a dataframe containing the file locations of each track in the library

In [5]:
pdf_library_paths = pd.read_parquet(path_library_parquet)

In [6]:
pdf_library_paths.head(5)

,id,path,name,artist
0,4822,/media/emily/DJ_Backup/Tracks - A/2025-05-18/7...,Lunch Box,Doll Factory
1,1443,/media/emily/DJ_Backup/Tracks - A/03 - Can I P...,Can I Play with Madness (2015 Remaster),Iron Maiden
2,4976,/media/emily/DJ_Backup/Tracks - A/2025-10-25/0...,Military Fashion Show (Club Hit),And One
3,2149,/media/emily/DJ_Backup/Tracks - A/2022-11-07/0...,Burn,The Cure
4,91,/media/emily/DJ_Backup/Tracks - A/Front 242 - ...,Headhunter,Front 242


## Define function enabling parallelized track analysis

In [7]:
def process_song_from_song_library(filename, track_id, sampling_rate, hop_length):
    y_tick_labels = librosa.key_to_notes('C:major')  # we compute this every time for now, neither efficient nor too costly

    # there is a file not found error I need to debug later
    try:
        y, sr = librosa.load(filename, sr = sampling_rate, mono = True)
    except:
        return None
    
    y_harmonic, y_percussive = librosa.effects.hpss(y)
    results_list = []
    for octave_number in range(0, 9):
        chromagram = process_octave(y_harmonic, octave_number, sr = sampling_rate, hop_length = hop_length)
        df = pd.DataFrame(chromagram.T)
        df.columns = [x + str(octave_number) for x in y_tick_labels]
        results_list.append(df)
    df_all_octaves = pd.concat(results_list, axis = 1)
    df_all_octaves['id'] = track_id
    return df_all_octaves

## Mathematically process track library to extract chromagram features

We obtain a chromagram vector that contains a separate value for every note on the piano, per time step.

Runs in parallel.

NOTE:  Using the 'capture' function suppresses warnings. Later I need to figure out why these warnings happen (they occur for an extremely small minority of tracks so I am ignoring this matter for now).

In [8]:
%%capture

args_list = []
for i, row in pdf_library_paths.iterrows():
    args_list.append((row['path'], row['id'], sampling_rate, hop_length))

with multiprocessing.Pool(processes = 50) as pool:
    results = pool.starmap(process_song_from_song_library, args_list)

In [9]:
len(results)

193

## Assemble track library feature vectors into a DataFrame

In [10]:
pdf_library = pd.concat(results, ignore_index = True)

In [11]:
pdf_library.head(3)

,C0,C♯0,D0,D♯0,E0,F0,F♯0,G0,G♯0,A0,...,D♯8,E8,F8,F♯8,G8,G♯8,A8,A♯8,B8,id
0,0.365672,0.197271,0.121671,0.209549,0.162165,0.180466,0.208330,0.303231,0.440096,0.154227,...,0.216138,0.267648,0.238951,0.305784,0.232654,0.280719,0.301721,0.144777,0.349615,4822
1,0.364031,0.191658,0.121127,0.201516,0.160385,0.177447,0.210006,0.303015,0.435153,0.160629,...,0.211462,0.264455,0.235790,0.320331,0.249778,0.288821,0.301595,0.142021,0.349249,4822
2,0.362304,0.185688,0.120480,0.193305,0.158500,0.174756,0.210761,0.303114,0.430160,0.168311,...,0.207521,0.260634,0.232011,0.332949,0.266672,0.296880,0.302558,0.140138,0.347316,4822


## QA

In [12]:
len(pdf_library.index)

22565

In [13]:
len(pdf_library.dropna().index)

22565

In [14]:
len(pdf_library['id'].unique())

185

## Remove rows with NaN values

I don't expect there to be any, and if the QA procedure above finds any, we need to figure out why.

In [15]:
pdf_library.dropna(inplace = True)

## Identify the column names for the notes

In [16]:
pitch_columns = [x for x in pdf_library.columns if x != 'id']

## Initiate a Spark session

In [17]:
conf = (
    SparkConf()
    .setAppName('AnalyzedTrackLibrary')
    .set('spark.executor.memory', spark_memory)
    .set('spark.driver.memory', spark_memory)
    .set('spark.driver.maxResultSize', spark_memory)
)

spark = SparkSession.builder.config(conf = conf).getOrCreate()

26/03/19 16:55:22 WARN Utils: Your hostname, emily-MS-7B96 resolves to a loopback address: 127.0.1.1; using 192.168.1.99 instead (on interface eno1)
26/03/19 16:55:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/19 16:55:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Convert the Pandas DF to a Spark DF and collapse value columns to a vector

In [18]:
sdf_library = (
    spark
    .createDataFrame(pdf_library)
    .orderBy('id')
    .repartition('id')
    .withColumn('array_library', F.array(*pitch_columns))
    .select('array_library', 'id')
)

In [19]:
sdf_library.show(5)

26/03/19 16:55:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+----+
|       array_library|  id|
+--------------------+----+
|[0.37314572930336...|4519|
|[0.37303182482719...|4519|
|[0.37311378121376...|4519|
|[0.37323668599128...|4519|
|[0.37320202589035...|4519|
+--------------------+----+
only showing top 5 rows



## Write the Spark DataFrame to a Parquet file

In [20]:
path_library_output = output_directory + '/library_vectors_spark_sr_' + str(sampling_rate) + '_hl_' + str(hop_length) + '.parquet'
sdf_library.write.mode('overwrite').parquet(path_library_output)

26/03/19 16:55:36 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


## Close the Spark session

In [21]:
spark.stop()